# AI-Based Smart Home Energy Consumption (Notebook)

Converted from `notebooks/python.py` and split into multiple runnable cells.

This version fixes path handling so it works in Jupyter (no `__file__`).


In [1]:
# SECTION 1: Imports + notebook-friendly paths
from pathlib import Path
import warnings

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

from statsmodels.tsa.arima.model import ARIMA

# Optional XGBoost
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("[INFO] XGBoost not installed — skipping XGBoost model.")

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# In notebooks, avoid GUI backends; we write plots to disk.
matplotlib.use("Agg")

cwd = Path.cwd().resolve()
if (cwd / "dataset").exists() and (cwd / "models").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "dataset").exists() and (cwd.parent / "models").exists():
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = cwd

PLOT_DIR = REPO_ROOT / "notebooks" / "plots"
MODEL_DIR = REPO_ROOT / "models"
DATA_PATH = REPO_ROOT / "dataset" / "energydata_complete.csv"

PLOT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def save_plot(fig, name: str) -> None:
    path = PLOT_DIR / name
    fig.savefig(str(path), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  ✔ Plot saved → {path}")

print("REPO_ROOT:", REPO_ROOT)
print("DATA_PATH:", DATA_PATH)
print("PLOT_DIR:", PLOT_DIR)
print("MODEL_DIR:", MODEL_DIR)


REPO_ROOT: C:\Users\krish\Documents\energy\notebooks
DATA_PATH: C:\Users\krish\Documents\energy\notebooks\dataset\energydata_complete.csv
PLOT_DIR: C:\Users\krish\Documents\energy\notebooks\notebooks\plots
MODEL_DIR: C:\Users\krish\Documents\energy\notebooks\models


In [2]:
# SECTION 2: Data loading
df = None
for enc in ["utf-8", "latin-1", "cp1252"]:
    try:
        df = pd.read_csv(str(DATA_PATH), encoding=enc)
        print(f"Loaded with encoding: {enc}")
        break
    except UnicodeDecodeError:
        continue

if df is None:
    raise RuntimeError(f"Failed to load dataset at: {DATA_PATH}")

df.columns = df.columns.str.strip().str.replace("\u00b0", "°")
print("Shape:", df.shape)
df.head()


Loaded with encoding: utf-8
Shape: (100000, 8)


,Home ID,Appliance Type,Energy Consumption (kWh),Time,Date,Outdoor Temperature (°C),Season,Household Size
0,94,Fridge,0.20,21:12,2023-12-02,-1.0,Fall,2
1,435,Oven,0.23,20:11,2023-08-06,31.1,Summer,5
2,466,Dishwasher,0.32,06:39,2023-11-21,21.3,Fall,3
3,496,Heater,3.92,21:56,2023-01-21,-4.2,Winter,1
4,137,Microwave,0.44,04:31,2023-08-26,34.5,Summer,5


In [3]:
# SECTION 3: EDA + time features
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values ✔")

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df["Energy Consumption (kWh)"], bins=50, kde=True, ax=ax, color="#3498db")
ax.set_title("Distribution of Energy Consumption (kWh)", fontsize=14)
ax.set_xlabel("Energy Consumption (kWh)")
save_plot(fig, "01_energy_distribution.png")

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(x="Appliance Type", y="Energy Consumption (kWh)", data=df, ax=ax, palette="Set2")
ax.set_title("Energy Consumption by Appliance Type", fontsize=14)
plt.xticks(rotation=45, ha="right")
save_plot(fig, "02_appliance_boxplot.png")

fig, ax = plt.subplots(figsize=(8, 5))
season_avg = df.groupby("Season")["Energy Consumption (kWh)"].mean().sort_values(ascending=False)
season_avg.plot(kind="bar", color=["#e74c3c", "#f39c12", "#2ecc71", "#3498db"], ax=ax)
ax.set_title("Average Energy Consumption by Season", fontsize=14)
ax.set_ylabel("Avg Energy (kWh)")
plt.xticks(rotation=0)
save_plot(fig, "03_season_bar.png")

fig, ax = plt.subplots(figsize=(10, 8))
numeric_cols = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_cols.corr(), annot=True, cmap="coolwarm", fmt=".2f", ax=ax, linewidths=0.5)
ax.set_title("Correlation Heatmap", fontsize=14)
save_plot(fig, "04_correlation_heatmap.png")

df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], errors="coerce")
df["Hour"] = df["Datetime"].dt.hour
df["Day"] = df["Datetime"].dt.day
df["Month"] = df["Datetime"].dt.month
df["Weekday"] = df["Datetime"].dt.weekday

fig, ax = plt.subplots(figsize=(10, 5))
hourly = df.groupby("Hour")["Energy Consumption (kWh)"].mean()
ax.plot(hourly.index, hourly.values, marker="o", color="#9b59b6", linewidth=2)
ax.set_title("Average Energy Consumption by Hour of Day", fontsize=14)
ax.set_xlabel("Hour")
ax.set_ylabel("Avg Energy (kWh)")
ax.set_xticks(range(0, 24))
save_plot(fig, "05_hourly_trend.png")

fig, ax = plt.subplots(figsize=(10, 5))
monthly = df.groupby("Month")["Energy Consumption (kWh)"].mean()
ax.bar(monthly.index, monthly.values, color="#1abc9c")
ax.set_title("Average Energy Consumption by Month", fontsize=14)
ax.set_xlabel("Month")
ax.set_ylabel("Avg Energy (kWh)")
ax.set_xticks(range(1, 13))
save_plot(fig, "06_monthly_trend.png")

fig, ax = plt.subplots(figsize=(10, 5))
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
weekday_avg = df.groupby("Weekday")["Energy Consumption (kWh)"].mean()
ax.bar(weekday_avg.index, weekday_avg.values, color="#e67e22", tick_label=day_names)
ax.set_title("Average Energy Consumption by Day of Week", fontsize=14)
ax.set_ylabel("Avg Energy (kWh)")
save_plot(fig, "07_weekday_trend.png")

temp_col = [c for c in df.columns if "Temperature" in c][0]
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df[temp_col], df["Energy Consumption (kWh)"], alpha=0.1, s=5, color="#e74c3c")
ax.set_title("Outdoor Temperature vs Energy Consumption", fontsize=14)
ax.set_xlabel(temp_col)
ax.set_ylabel("Energy Consumption (kWh)")
save_plot(fig, "08_temp_vs_energy.png")


No missing values ✔
  ✔ Plot saved → C:\Users\krish\Documents\energy\notebooks\notebooks\plots\01_energy_distribution.png
  ✔ Plot saved → C:\Users\krish\Documents\energy\notebooks\notebooks\plots\02_appliance_boxplot.png
  ✔ Plot saved → C:\Users\krish\Documents\energy\notebooks\notebooks\plots\03_season_bar.png
  ✔ Plot saved → C:\Users\krish\Documents\energy\notebooks\notebooks\plots\04_correlation_heatmap.png
  ✔ Plot saved → C:\Users\krish\Documents\energy\notebooks\notebooks\plots\05_hourly_trend.png
  ✔ Plot saved → C:\Users\krish\Documents\energy\notebooks\notebooks\plots\06_monthly_trend.png
  ✔ Plot saved → C:\Users\krish\Documents\energy\notebooks\notebooks\plots\07_weekday_trend.png
  ✔ Plot saved → C:\Users\krish\Documents\energy\notebooks\notebooks\plots\08_temp_vs_energy.png


In [4]:
# SECTION 4: Data preprocessing
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(include=["object"]).columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

df.dropna(subset=["Datetime"], inplace=True)
print("Shape after cleaning:", df.shape)
print("Seasons:", df["Season"].unique())

df_before_fe = df.copy()


Shape after cleaning: (100000, 13)
Seasons: ['Fall' 'Summer' 'Winter' 'Spring']


In [5]:
# SECTION 5: Feature engineering
df["Peak_Hour"] = df["Hour"].apply(lambda h: 1 if 18 <= h <= 22 else 0)
df["Night_Usage"] = df["Hour"].apply(lambda h: 1 if h >= 23 or h <= 5 else 0)
df["Temp_Squared"] = df[temp_col] ** 2
df["Temp_x_HouseholdSize"] = df[temp_col] * df["Household Size"]

df = df.sort_values("Datetime").reset_index(drop=True)
df["Rolling_Avg_3"] = (
    df.groupby("Home ID")["Energy Consumption (kWh)"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

for lag in [1, 2, 3]:
    df[f"Lag_{lag}"] = (
        df.groupby("Home ID")["Energy Consumption (kWh)"]
        .transform(lambda x: x.shift(lag))
    )

df.fillna(0, inplace=True)
print("Shape after FE:", df.shape)


Shape after FE: (100000, 21)


In [6]:
# Helper: prepare features
def prepare_features(dataframe, label="Energy Consumption (kWh)"):
    drop_cols = [label, "Datetime", "Date", "Time"]
    drop_cols = [c for c in drop_cols if c in dataframe.columns]

    X = dataframe.drop(columns=drop_cols)
    y = dataframe[label]

    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    ct = ColumnTransformer(
        [
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
            ("scaler", StandardScaler(), num_cols),
        ],
        remainder="drop",
    )

    X_transformed = ct.fit_transform(X)

    ohe_names = (
        ct.named_transformers_["onehot"].get_feature_names_out(cat_cols).tolist()
        if cat_cols else []
    )
    feature_names = ohe_names + num_cols

    X_train, X_test, y_train, y_test = train_test_split(
        X_transformed, y, test_size=0.2, random_state=42
    )

    return X_train, X_test, y_train, y_test, feature_names, ct


In [7]:
# SECTION 6: PCA
X_tr_after, X_te_after, y_tr_after, y_te_after, feat_names_after, ct_after = prepare_features(df)

pca_full = PCA()
pca_full.fit(X_tr_after)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cum_var) + 1), cum_var, marker="o", color="#8e44ad")
ax.axhline(y=0.95, color="red", linestyle="--", label="95% threshold")
ax.set_title("PCA — Cumulative Explained Variance", fontsize=14)
ax.set_xlabel("Number of Components")
ax.set_ylabel("Cumulative Explained Variance")
ax.legend()
save_plot(fig, "09_pca_variance.png")

n_components_95 = int(np.argmax(cum_var >= 0.95) + 1)
print("Components for 95% variance:", n_components_95)

pca = PCA(n_components=n_components_95)
X_tr_pca = pca.fit_transform(X_tr_after)
X_te_pca = pca.transform(X_te_after)

rf_pca = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_pca.fit(X_tr_pca, y_tr_after)
pred_pca = rf_pca.predict(X_te_pca)
r2_pca = r2_score(y_te_after, pred_pca)

rf_no_pca = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_no_pca.fit(X_tr_after, y_tr_after)
pred_no_pca = rf_no_pca.predict(X_te_after)
r2_no_pca = r2_score(y_te_after, pred_no_pca)

print(f"R² WITHOUT PCA: {r2_no_pca:.4f}")
print(f"R² WITH    PCA: {r2_pca:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Without PCA", "With PCA"], [r2_no_pca, r2_pca], color=["#3498db", "#e74c3c"])
ax.set_title("Random Forest R² — Before vs After PCA", fontsize=13)
ax.set_ylabel("R² Score")
for bar, val in zip(bars, [r2_no_pca, r2_pca]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, f"{val:.4f}", ha="center", fontsize=11)
save_plot(fig, "10_pca_comparison.png")


  ✔ Plot saved → C:\Users\krish\Documents\energy\notebooks\notebooks\plots\09_pca_variance.png
Components for 95% variance: 17
R² WITHOUT PCA: 0.9511
R² WITH    PCA: 0.9913
  ✔ Plot saved → C:\Users\krish\Documents\energy\notebooks\notebooks\plots\10_pca_comparison.png


In [8]:
# Helper: train/evaluate models
def train_evaluate_models(X_train, X_test, y_train, y_test):
    models = {
        "Linear Regression": LinearRegression(),
        "Decision Tree": DecisionTreeRegressor(max_depth=10, min_samples_split=10, random_state=42),
        "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_split=10, random_state=42),
        "KNN": KNeighborsRegressor(n_neighbors=5),
    }
    if HAS_XGBOOST:
        models["XGBoost"] = XGBRegressor(
            n_estimators=200, max_depth=6, learning_rate=0.05,
            subsample=0.8, random_state=42, verbosity=0
        )

    results = []
    trained_models = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        mae = mean_absolute_error(y_test, preds)
        mse = mean_squared_error(y_test, preds)
        rmse = float(np.sqrt(mse))
        r2 = r2_score(y_test, preds)
        results.append({"Model": name, "MAE": mae, "MSE": mse, "RMSE": rmse, "R²": r2})
        trained_models[name] = model

    return pd.DataFrame(results), trained_models


In [9]:
# SECTION 7: Train BEFORE feature engineering
X_tr_before, X_te_before, y_tr_before, y_te_before, feat_names_before, ct_before = prepare_features(df_before_fe)
results_before, models_before = train_evaluate_models(X_tr_before, X_te_before, y_tr_before, y_te_before)
results_before


,Model,MAE,MSE,RMSE,R²
0,Linear Regression,0.475998,0.341065,0.584007,0.757515
1,Decision Tree,0.482949,0.360273,0.600228,0.743858
2,Random Forest,0.475865,0.341357,0.584258,0.757306
3,KNN,0.505018,0.404787,0.636228,0.712210
4,XGBoost,0.476661,0.343675,0.586238,0.755659


In [10]:
# SECTION 8: Train AFTER feature engineering
results_after, models_after = train_evaluate_models(X_tr_after, X_te_after, y_tr_after, y_te_after)
results_after


,Model,MAE,MSE,RMSE,R²
0,Linear Regression,0.124142,0.061991,0.248979,0.956219
1,Decision Tree,0.239200,0.103940,0.322397,0.926593
2,Random Forest,0.189072,0.069293,0.263236,0.951062
3,KNN,0.457146,0.331221,0.575518,0.766075
4,XGBoost,0.091615,0.016803,0.129626,0.988133


In [11]:
# SECTION 9: Comparison table + plot
comparison = results_before[["Model", "R²"]].rename(columns={"R²": "R²_Before"}).merge(
    results_after[["Model", "R²"]].rename(columns={"R²": "R²_After"}),
    on="Model",
)
comparison["Improvement"] = comparison["R²_After"] - comparison["R²_Before"]
comparison

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison))
width = 0.35
ax.bar(x - width / 2, comparison["R²_Before"], width, label="Before FE", color="#3498db")
ax.bar(x + width / 2, comparison["R²_After"], width, label="After FE", color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(comparison["Model"], rotation=20, ha="right")
ax.set_ylabel("R² Score")
ax.set_title("Model R² — Before vs After Feature Engineering", fontsize=14)
ax.legend()
ax.set_ylim(0, max(comparison["R²_After"].max(), comparison["R²_Before"].max()) * 1.15)
save_plot(fig, "11_model_comparison.png")


  ✔ Plot saved → C:\Users\krish\Documents\energy\notebooks\notebooks\plots\11_model_comparison.png


In [12]:
# SECTION 10: KMeans clustering
home_agg = df.groupby("Home ID").agg(
    Avg_Consumption=("Energy Consumption (kWh)", "mean"),
    Total_Consumption=("Energy Consumption (kWh)", "sum"),
    Household_Size=("Household Size", "first"),
    Peak_Usage_Ratio=("Peak_Hour", "mean"),
    Night_Usage_Ratio=("Night_Usage", "mean"),
    Avg_Temperature=(temp_col, "mean"),
).reset_index()

cluster_features = [
    "Avg_Consumption",
    "Total_Consumption",
    "Peak_Usage_Ratio",
    "Night_Usage_Ratio",
    "Avg_Temperature",
]
scaler_cluster = StandardScaler()
X_cluster = scaler_cluster.fit_transform(home_agg[cluster_features])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
home_agg["Cluster"] = kmeans.fit_predict(X_cluster)
home_agg.head()


,Home ID,Avg_Consumption,Total_Consumption,Household_Size,Peak_Usage_Ratio,Night_Usage_Ratio,Avg_Temperature,Cluster
0,1,1.539147,324.76,2,0.165877,0.293839,13.632701,2
1,2,1.490526,311.52,1,0.200957,0.291866,15.357895,2
2,3,1.537315,332.06,5,0.300926,0.236111,13.097685,0
3,4,1.458667,262.56,2,0.155556,0.333333,13.543333,2
4,5,1.435121,297.07,4,0.236715,0.256039,13.954106,1


In [13]:
# SECTION 11: ARIMA forecasting (30 days)
daily = (
    df.groupby(df["Datetime"].dt.date)["Energy Consumption (kWh)"]
    .sum()
    .reset_index()
)
daily.columns = ["Date", "Daily_Consumption"]
daily["Date"] = pd.to_datetime(daily["Date"])
daily = daily.sort_values("Date").set_index("Date")

arima_model = ARIMA(daily["Daily_Consumption"], order=(5, 1, 2))
arima_result = arima_model.fit()

forecast = arima_result.forecast(steps=30)
forecast_index = pd.date_range(start=daily.index[-1] + pd.Timedelta(days=1), periods=30)
forecast_series = pd.Series(forecast.values, index=forecast_index)

forecast_series.to_csv(str(MODEL_DIR / "forecast_30days.csv"), header=["Forecast"])
print("Saved:", MODEL_DIR / "forecast_30days.csv")


Saved: C:\Users\krish\Documents\energy\notebooks\models\forecast_30days.csv


In [14]:
# SECTION 12: Save best model + transformers
best_model_name = results_after.loc[results_after["R²"].idxmax(), "Model"]
best_model = models_after[best_model_name]
best_r2 = float(results_after["R²"].max())

joblib.dump(best_model, str(MODEL_DIR / "best_model.pkl"))
joblib.dump(ct_after, str(MODEL_DIR / "column_transformer.pkl"))

metadata = {
    "best_model_name": best_model_name,
    "r2_score": best_r2,
    "n_features": len(feat_names_after),
    "feature_names": feat_names_after,
    "temp_column": temp_col,
}
joblib.dump(metadata, str(MODEL_DIR / "metadata.pkl"))

print("Saved model artifacts to:", MODEL_DIR)


Saved model artifacts to: C:\Users\krish\Documents\energy\notebooks\models


In [15]:
# SECTION 13: Prediction function + quick test
def predict_energy(temperature, appliance_type, household_size, hour, season):
    model = joblib.load(str(MODEL_DIR / "best_model.pkl"))
    ct = joblib.load(str(MODEL_DIR / "column_transformer.pkl"))
    meta = joblib.load(str(MODEL_DIR / "metadata.pkl"))
    temp_column = meta["temp_column"]

    row = {
        "Home ID": 0,
        "Appliance Type": appliance_type,
        temp_column: temperature,
        "Season": season,
        "Household Size": household_size,
        "Hour": hour,
        "Day": 15,
        "Month": 6,
        "Weekday": 2,
        "Peak_Hour": 1 if 18 <= hour <= 22 else 0,
        "Night_Usage": 1 if hour >= 23 or hour <= 5 else 0,
        "Temp_Squared": temperature ** 2,
        "Temp_x_HouseholdSize": temperature * household_size,
        "Rolling_Avg_3": 0.0,
        "Lag_1": 0.0,
        "Lag_2": 0.0,
        "Lag_3": 0.0,
    }

    input_df = pd.DataFrame([row])
    X_input = ct.transform(input_df)
    pred = model.predict(X_input)[0]
    return round(float(pred), 4)

predict_energy(temperature=25.0, appliance_type="Heater", household_size=4, hour=19, season="Winter")


1.6906